# Nemotron 3 Nano - Min/Max-LogProb SFT (clean rewrite, zero-loss-proof)

Full rewrite of the min-max SFT notebook. Prior versions logged **loss = 0.0**
forever. Two root causes, both fixed here:

1. **TRL `SFTTrainer` + `skip_prepare_dataset` mangled the label path** -> the
   loss saw all-`-100` labels -> `nll[mask].mean()` over an empty mask -> `nan`
   -> the silent `logits.sum()*0.0` guard turned it into a clean `0.0`.
   **Fix:** use vanilla `transformers.Trainer` with our own collator so
   `input_ids` / `labels` reach `compute_loss` verbatim.
2. **Silent non-finite guard hid everything.** **Fix:** fp32 reductions, sanitise
   non-finite logits and recompute (real gradient, not a dead zero), and make any
   fallback *loud* with counters.

Plus a **PRE-FLIGHT cell**: one real forward+backward that asserts the loss is
finite & > 0.3 and that LoRA grads actually flow -- run it before burning compute,
so a zero-loss run is impossible to start unnoticed.

Loss menu via `LOSS_MODE`: `mean` | `minmax` | `topk_min` | `blend` | `branch`.

In [1]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0}


In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f51f4c4bd10>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

Training environment fixes applied.


In [4]:
import os

BASE_MODEL_NAME   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH     = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

OUTPUT_ROOT       = "outputs"
SFT_ADAPTER_DIR   = os.path.join(OUTPUT_ROOT, "minmax_logprob_adapter")
SUBMISSION_DIR    = os.path.join(OUTPUT_ROOT, "submission_minmax_logprob")
TB_LOG_DIR        = os.path.join(OUTPUT_ROOT, "tb_logs_minmax_logprob")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42

# ── compute-safety ──
SMOKE_TEST  = 0         # 1: tiny dry-run; flip to 0 for the real run
SMOKE_ROWS  = 64
SMOKE_STEPS = 8
SUBSET_N    = 3000       # None -> all rows
NUM_EPOCHS  = 2
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 8192     # cipher CoTs run ~8k tokens; boxed answer is at the END,
                         # so do NOT shrink below what fits the answer.
STRATIFIED_BATCHING = True

# ── loss ──
# mean | minmax | topk_min | blend | branch
LOSS_MODE = "blend"
TOPK_MIN  = 16
BLEND_ALPHA = 0.5
BRANCH_LOGPROB = 1.0
WARMUP_MEAN_STEPS = 0

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({"SMOKE_TEST": SMOKE_TEST, "SUBSET_N": SUBSET_N, "NUM_EPOCHS": NUM_EPOCHS,
       "TRAIN_MAX_LEN": TRAIN_MAX_LEN, "LOSS_MODE": LOSS_MODE, "TOPK_MIN": TOPK_MIN,
       "BLEND_ALPHA": BLEND_ALPHA, "BRANCH_LOGPROB": BRANCH_LOGPROB,
       "WARMUP_MEAN_STEPS": WARMUP_MEAN_STEPS})

{'SMOKE_TEST': 0, 'SUBSET_N': 3000, 'NUM_EPOCHS': 2, 'TRAIN_MAX_LEN': 8192, 'LOSS_MODE': 'blend', 'TOPK_MIN': 16, 'BLEND_ALPHA': 0.5, 'BRANCH_LOGPROB': 1.0, 'WARMUP_MEAN_STEPS': 0}


In [5]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

Torch: 2.10.0+cu128 CUDA: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False, load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"     # SFT loss wants right padding
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-03 09:41:54.661143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780479714.884907      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780479714.941312      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780479715.490450      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780479715.490463      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780479715.490464      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


## LoRA targets (RSLoRA r=32, sensitive-module priority)

DoRA intentionally OFF in this rewrite -- fewer moving parts while we confirm the
loss is healthy. Re-enable later if you want the +2-4pp. Grad path is hardened:
`enable_input_require_grads()` + `use_cache=False` so reentrant gradient
checkpointing actually propagates into the LoRA params.

In [7]:
from peft import LoraConfig, get_peft_model, TaskType
import re

linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)
matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"LoRA target regex matched {len(matched)} modules.")
if len(matched) == 0:
    sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
    raise RuntimeError(f"LoRA target_regex matched 0 modules. Sample: {sample}")

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
    target_modules=target_regex,
    task_type=TaskType.CAUSAL_LM,
    use_rslora=True,
    use_dora=False,
)
model = get_peft_model(model, lora_config)

# ── grad-path hardening (the difference between real training and dead zeros) ──
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()   # reentrant ckpt needs an input requiring grad
try:
    model.config.use_cache = False
except Exception:
    pass
model.train()

model.print_trainable_parameters()
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
# print(f"[audit] {len(trainable)} trainable tensors  total={n_train/1e6:.1f}M")
# assert 50_000_000 <= n_train <= 400_000_000, \
#     f"[audit] trainable count {n_train/1e6:.1f}M outside [50M,400M] -- inspect target_regex."

LoRA target regex matched 46 modules.
trainable params: 9,420,800 || all params: 31,587,358,144 || trainable%: 0.0298


## System prompt (identical at train + eval)

In [8]:
SYSTEM_PROMPT = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses.
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F carefully. State the source and target base. No prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units. Round only at the end.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme and direction. Transform \
one character at a time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations; or apply the defined transformation rule literally. Keep equations balanced.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

print(f"SYSTEM_PROMPT chars: {len(SYSTEM_PROMPT)}")

SYSTEM_PROMPT chars: 2417


## Dataset prep + assistant-only masking

Rebuild every target as `<think>\n{reasoning}\n</think>\n\boxed{answer}` (closing
tag guaranteed, single trailing box from the answer column). Hard-fails on any
malformed target so a broken corpus can't silently train.

In [9]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")

def _find(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n in low: return low[n]
    return None

PROMPT_COL = _find(df_sft.columns, ["prompt", "question", "problem", "input"])
ANSWER_COL = _find(df_sft.columns, ["answer", "solution", "label", "target", "final_answer"])
COT_COL    = _find(df_sft.columns, ["cot", "reasoning", "think", "generated_cot",
                                    "response", "completion", "rationale", "output"])
TYPE_COL   = _find(df_sft.columns, ["type", "category", "puzzle_type", "task_type"])
print(f"Detected -> prompt={PROMPT_COL!r}  answer={ANSWER_COL!r}  cot={COT_COL!r}  type={TYPE_COL!r}")
if PROMPT_COL is None:
    raise ValueError(f"No prompt-like column in {list(df_sft.columns)}")

df_sft = df_sft.dropna(subset=[PROMPT_COL]).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    df_sft = df_sft.head(SMOKE_ROWS).reset_index(drop=True)
    print(f"[SMOKE] using {len(df_sft)} rows")
elif SUBSET_N is not None:
    df_sft = df_sft.head(SUBSET_N).reset_index(drop=True)
    print(f"[REAL] using subset of {len(df_sft)} rows")
else:
    print(f"[REAL] using all {len(df_sft)} rows")

def build_assistant_text(row):
    """Canonical target: <think>\n{reasoning}\n</think>\n\\boxed{ans}."""
    ans = "" if ANSWER_COL is None else str(row[ANSWER_COL]).strip()
    cot = "" if COT_COL is None else str(row.get(COT_COL, "") or "").strip()
    cot = cot.replace("<think>", "").replace("</think>", "").strip()
    cot = re.sub(r'(?im)^.*I will (now )?(put|return) .*\\boxed\{\}.*$', '', cot)
    cot = re.sub(r'(?im)^.*The answer .*\\boxed\{[^}]*\}.*$', '', cot)
    cot = re.sub(r'\\boxed\{[^{}]*\}', '', cot)
    cot = re.sub(r'\n{3,}', '\n\n', cot).strip()
    think = cot if cot else "Work through the problem step by step."
    return f"<think>\n{think}\n</think>\n\\boxed{{{ans}}}"

records, record_types = [], []
for _, row in df_sft.iterrows():
    records.append({
        "system":    SYSTEM_PROMPT,
        "user":      str(row[PROMPT_COL]) + PROMPT_SUFFIX,
        "assistant": build_assistant_text(row),
    })
    record_types.append(str(row[TYPE_COL]) if TYPE_COL else "unknown")

# Instead of raising an error, filter out malformed records
_bad = [i for i, r in enumerate(records)
        if "</think>" not in r["assistant"]
        or r["assistant"].count("\\boxed{") != 1
        or not r["assistant"].rstrip().endswith("}")]
if _bad:
    print(f"[WARNING] Skipping {len(_bad)} malformed targets (first 5 indices: {_bad[:5]})")
    bad_set = set(_bad)
    records = [r for i, r in enumerate(records) if i not in bad_set]
    record_types = [t for i, t in enumerate(record_types) if i not in bad_set]
else:
    print(f"[format] all {len(records)} targets well-formed.")

if len(records) == 0:
    raise RuntimeError("No valid records left after filtering malformed targets.")

raw_ds = HFDataset.from_list(records)
print("Type distribution:", dict(pd.Series(record_types).value_counts().head(10).to_dict()))
print("\n--- sample target TAIL ---\n", records[0]["assistant"][-160:])

SFT data: 7830 rows.  Columns: ['id', 'prompt', 'answer', 'type', 'generated_cot']
Detected -> prompt='prompt'  answer='answer'  cot='generated_cot'  type='type'
[REAL] using subset of 3000 rows
[WARNING] Skipping 40 malformed targets (first 5 indices: [58, 123, 236, 249, 293])
Type distribution: {'bit_manipulation': 661, 'cipher': 623, 'unit_conversion': 415, 'gravity': 379, 'numeral': 291, 'equation_numeric_deduce': 259, 'cryptarithm_deduce': 222, 'cryptarithm_guess': 63, 'equation_numeric_guess': 47}

--- sample target TAIL ---
 3 = NOT(0) = 1
3 NOT4 = NOT(1) = 0
4 NOT5 = NOT(1) = 0
5 NOT6 = NOT(0) = 1
6 OR-NOT07 = OR(0,NOT(1)) = 0
7 OR-NOT10 = OR(0,NOT(0)) = 1
</think>
\boxed{01100101}


In [10]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)
    full_text   = render(full_msgs,   False)
    prefix_text = render(prefix_msgs, True)
    full_ids   = tokenizer(full_text,   add_special_tokens=False, truncation=True,
                           max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}

tokenized_ds = raw_ds.map(tokenize_with_assistant_mask,
                          remove_columns=raw_ds.column_names,
                          desc="Tokenize + assistant mask")

_kept_rows, _kept_types, _n_all_masked = [], [], 0
for i, ex in enumerate(tokenized_ds):
    if any(t != -100 for t in ex["labels"]):
        _kept_rows.append(ex); _kept_types.append(record_types[i])
    else:
        _n_all_masked += 1
tokenized_ds = HFDataset.from_list(_kept_rows)
record_types = _kept_types
print(f"Kept {len(tokenized_ds)} rows (dropped {_n_all_masked} fully-masked).")

if len(tokenized_ds) == 0:
    raise RuntimeError("tokenized_ds EMPTY -> every row fully masked. Masking broken.")

_ex0 = tokenized_ds[0]
_un = [t for t, l in zip(_ex0["input_ids"], _ex0["labels"]) if l != -100]
print(f"[mask-check] row0 total={len(_ex0['input_ids'])} unmasked={len(_un)}")
print("[mask-check] decoded unmasked:", repr(tokenizer.decode(_un)[:200]))
assert "boxed" in tokenizer.decode(_un), "[mask-check] no boxed in unmasked span -> misaligned."

import numpy as np
_lens = np.array([len(x["input_ids"]) for x in tokenized_ds])
_ul   = np.array([sum(1 for l in x["labels"] if l != -100) for x in tokenized_ds])
print(f"len min={_lens.min()} mean={_lens.mean():.0f} p90={int(np.percentile(_lens,90))} max={_lens.max()}")
print(f"unmasked/seq min={_ul.min()} mean={_ul.mean():.0f} max={_ul.max()}")

Tokenize + assistant mask:   0%|          | 0/2960 [00:00<?, ? examples/s]

Kept 2960 rows (dropped 0 fully-masked).
[mask-check] row0 total=8134 unmasked=7325
[mask-check] decoded unmasked: 'We need to deduce the transformation by matching the example outputs.\n\nOutput 0: 01001110\n0 0\n1 1\n2 0\n3 0\n4 1\n5 1\n6 1\n7 0\n\nOutput 1: 00010110\n0 0\n1 0\n2 0\n3 1\n4 0\n5 1\n6 1\n7 0\n\nOutput 2: 01000110\n0 0\n1 '
len min=1051 mean=4375 p90=7513 max=8192
unmasked/seq min=450 mean=3700 max=7421


In [11]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels/attention_mask; preserves -100 on prompt tokens."""
    def __init__(self, tokenizer, label_pad_id=-100):
        self.pad_id = tokenizer.pad_token_id
        self.label_pad_id = label_pad_id
    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attn = [], [], []
        for f in features:
            ids = list(f["input_ids"]); lab = list(f["labels"])
            pad = maxlen - len(ids)
            input_ids.append(ids + [self.pad_id] * pad)
            labels.append(lab + [self.label_pad_id] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready.")

Collator ready.


## Loss helpers + vanilla `transformers.Trainer` subclass

No TRL `SFTTrainer` -> no dataset-prep magic that drops labels. Per-token NLL via
fused `cross_entropy(reduction="none")`; reductions in fp32; non-finite logits are
sanitised and recomputed (real gradient), with loud counters.

In [12]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"

import torch, torch._dynamo, math, random
import torch.nn.functional as F
from collections import defaultdict
from torch.utils.data import DataLoader, Sampler
from transformers import Trainer, TrainingArguments
torch._dynamo.config.disable = True
torch._dynamo.reset()


def advanced_loss(nll_flat, mask, mode, topk_min, blend_alpha, branch_logprob):
    """nll_flat:(B,T) fp32 per-token NLL (0 where ignored). mask:(B,T) bool."""
    if mode == "mean":
        return nll_flat[mask].mean()
    if mode == "minmax":
        m = nll_flat.masked_fill(~mask, float("-inf"))
        worst, _ = m.max(dim=-1)
        return worst[mask.any(dim=-1)].mean()
    if mode in ("topk_min", "blend"):
        m = nll_flat.masked_fill(~mask, float("-inf"))
        k = min(topk_min, nll_flat.size(1))
        topk_vals, _ = torch.topk(m, k=k, dim=-1)
        valid_count = mask.sum(dim=-1).clamp(min=1)
        eff_k = torch.minimum(torch.full_like(valid_count, k), valid_count)
        real = torch.where(torch.isfinite(topk_vals), topk_vals, torch.zeros_like(topk_vals))
        per_seq = real.sum(dim=-1) / eff_k.to(real.dtype)
        tk = per_seq[mask.any(dim=-1)].mean()
        if mode == "topk_min":
            return tk
        return blend_alpha * nll_flat[mask].mean() + (1.0 - blend_alpha) * tk
    if mode == "branch":
        nm = nll_flat * mask.to(nll_flat.dtype)
        w = (nm / branch_logprob).clamp(max=1.0) * mask.to(nm.dtype)
        return (nm * w).sum() / w.sum().clamp(min=1.0)
    raise ValueError(f"bad loss_mode={mode}")


def per_token_nll(shift_logits, shift_labels, counters=None):
    B, T, V = shift_logits.shape
    nll = F.cross_entropy(shift_logits.reshape(-1, V), shift_labels.reshape(-1),
                          ignore_index=-100, reduction="none").view(B, T)
    if not torch.isfinite(nll).all():
        if counters is not None: counters["nonfinite"] += 1
        sl = torch.nan_to_num(shift_logits, nan=0.0, posinf=30.0, neginf=-30.0)
        nll = F.cross_entropy(sl.reshape(-1, V), shift_labels.reshape(-1),
                              ignore_index=-100, reduction="none").view(B, T)
        nll = torch.nan_to_num(nll, nan=0.0, posinf=30.0, neginf=0.0)
    return nll.float()


def build_stratified_index_order(labels, batch_size, seed):
    by = defaultdict(list)
    for idx, lab in enumerate(labels): by[lab].append(idx)
    rng = random.Random(seed)
    for v in by.values(): rng.shuffle(v)
    nb = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(nb)]
    order = list(range(nb)); rng.shuffle(order)
    a = 0
    for lab in sorted(by.keys()):
        for idx in by[lab]:
            batches[order[a % nb]].append(idx); a += 1
    out = [i for b in batches for i in b]
    if len(out) != len(labels): raise ValueError("stratified size mismatch")
    return out

class _OrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)


class AdvancedTrainer(Trainer):
    def __init__(self, *a, loss_mode="mean", topk_min=16, blend_alpha=0.5,
                 branch_logprob=1.0, warmup_mean_steps=0, stratified_order=None, **k):
        super().__init__(*a, **k)
        self.loss_mode = loss_mode; self.topk_min = topk_min
        self.blend_alpha = blend_alpha; self.branch_logprob = branch_logprob
        self.warmup_mean_steps = warmup_mean_steps
        self.stratified_order = stratified_order
        self._c = {"nonfinite": 0, "dead": 0, "dbg": 0}

    def _mode(self):
        return "mean" if self.state.global_step < self.warmup_mean_steps else self.loss_mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:].to(shift_logits.device)
        nll = per_token_nll(shift_logits, shift_labels, self._c)
        mask = (shift_labels != -100)
        mode = self._mode()
        loss = advanced_loss(nll, mask, mode, self.topk_min, self.blend_alpha, self.branch_logprob)

        if self._c["dbg"] < 3:
            self._c["dbg"] += 1
            nu = int(mask.sum())
            print(f"[loss-dbg step~{self.state.global_step}] mode={mode} loss={float(loss):.4f} "
                  f"unmasked={nu} nll[mn/mean/mx]={float(nll[mask].min()):.2f}/"
                  f"{float(nll[mask].mean()):.2f}/{float(nll[mask].max()):.2f} "
                  f"nonfinite={self._c['nonfinite']}")

        if not torch.isfinite(loss):
            self._c["dead"] += 1
            if self._c["dead"] <= 5 or self._c["dead"] % 50 == 0:
                print(f"[loss-WARN] non-finite loss step {self.state.global_step} "
                      f"(dead={self._c['dead']}, nonfinite_logits={self._c['nonfinite']}). "
                      f"If frequent: lower TRAIN_MAX_LEN or batch size.")
            loss = (logits.float().sum() * 0.0).requires_grad_(True)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        kw = dict(batch_size=self.args.per_device_train_batch_size,
                  sampler=_OrderSampler(self.stratified_order),
                  collate_fn=self.data_collator,
                  num_workers=self.args.dataloader_num_workers,
                  pin_memory=self.args.dataloader_pin_memory,
                  drop_last=self.args.dataloader_drop_last)
        return DataLoader(self.train_dataset, **kw)

print("AdvancedTrainer defined (vanilla transformers.Trainer).")

AdvancedTrainer defined (vanilla transformers.Trainer).


## PRE-FLIGHT: prove the loss is real before training

One forward+backward on a real collated batch. Asserts:
- the collated batch has unmasked labels (else loss is 0 by construction),
- the loss is finite and **> 0.3** (a fresh LoRA can't already be near-perfect),
- at least one LoRA tensor gets a **non-zero gradient** (grad path is alive).

If any assert fires, training would have produced zeros -- fix here, don't launch.

In [13]:
import torch, torch.nn.functional as F

_dl = DataLoader(tokenized_ds.select(range(min(2, len(tokenized_ds)))),
                 batch_size=1, collate_fn=data_collator)
_batch = next(iter(_dl))
_n_un = int((_batch["labels"] != -100).sum())
print(f"[preflight] collated batch unmasked labels = {_n_un}")
assert _n_un > 0, "Collated batch has ZERO unmasked labels -> loss would be 0. Collator/masking broken."

model.train()
try: model.config.use_cache = False
except Exception: pass
_batch = {k: v.to(model.device) for k, v in _batch.items()}
_labels = _batch.pop("labels")

model.zero_grad(set_to_none=True)
_out = model(**_batch)
_logits = _out.logits
print(f"[preflight] logits finite = {bool(torch.isfinite(_logits).all())}  shape={tuple(_logits.shape)}")

_sl = _logits[:, :-1, :]
_lab = _labels[:, 1:].to(_sl.device)
_nll = F.cross_entropy(_sl.reshape(-1, _sl.size(-1)), _lab.reshape(-1), ignore_index=-100)
print(f"[preflight] mean NLL = {float(_nll):.4f}")
assert torch.isfinite(_nll), "Pre-flight loss non-finite -> forward emits NaN/Inf (lower TRAIN_MAX_LEN/batch)."
assert _nll.item() > 0.3, f"Pre-flight loss {_nll.item():.4f} suspiciously low -> labels likely degenerate."

_nll.backward()
_grad_tensors = [(n, float(p.grad.abs().sum()))
                 for n, p in model.named_parameters()
                 if p.requires_grad and p.grad is not None and p.grad.abs().sum() > 0]
print(f"[preflight] LoRA tensors with non-zero grad = {len(_grad_tensors)}")
assert len(_grad_tensors) > 0, \
    "NO LoRA gradient flowed -> grad path broken (enable_input_require_grads / checkpointing)."
print("[preflight] example grad tensors:", [n for n, _ in _grad_tensors[:3]])
model.zero_grad(set_to_none=True)
del _out, _logits, _sl, _nll
torch.cuda.empty_cache()
print("\nPRE-FLIGHT PASSED: loss is real (>0.3) and LoRA grads flow. Safe to train.")

[preflight] collated batch unmasked labels = 7325
Unsloth: Will smartly offload gradients to save VRAM!
[preflight] logits finite = True  shape=(1, 8134, 131072)
[preflight] mean NLL = 0.4565


/tmp/ipykernel_65/3782211372.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"[preflight] mean NLL = {float(_nll):.4f}")


[preflight] LoRA tensors with non-zero grad = 46
[preflight] example grad tensors: ['base_model.model.backbone.layers.1.mixer.shared_experts.up_proj.lora_B.default.weight', 'base_model.model.backbone.layers.1.mixer.shared_experts.down_proj.lora_B.default.weight', 'base_model.model.backbone.layers.3.mixer.shared_experts.up_proj.lora_B.default.weight']

PRE-FLIGHT PASSED: loss is real (>0.3) and LoRA grads flow. Safe to train.


In [14]:
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1

args = TrainingArguments(
    output_dir                   = os.path.join(OUTPUT_ROOT, "minmax_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 8,
    learning_rate                = 8e-5,
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.95,
    bf16                         = True,
    gradient_checkpointing       = False,   # model already checkpoints (cell 8)
    remove_unused_columns        = False,   # keep input_ids/labels for compute_loss
    logging_steps                = 1 if SMOKE_TEST else 10,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    save_strategy                = "no" if SMOKE_TEST else "steps",
    save_steps                   = 100,
    save_total_limit             = 2,
    seed                         = SEED,
    dataloader_num_workers       = 2,
)

eff_batch = max(1, args.per_device_train_batch_size * args.gradient_accumulation_steps)
stratified_order = None
if STRATIFIED_BATCHING and len(set(record_types)) > 1:
    stratified_order = build_stratified_index_order(record_types, eff_batch, SEED)
    print(f"Stratified order built (eff_batch={eff_batch})")
else:
    print("Stratified batching off / single type -> default shuffle.")

print(f"Args ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'} max_steps={_max_steps} "
      f"epochs={NUM_EPOCHS} eff_batch={eff_batch} max_len={TRAIN_MAX_LEN} loss={LOSS_MODE}")

Stratified order built (eff_batch=8)
Args ready. mode=REAL max_steps=-1 epochs=2 eff_batch=8 max_len=8192 loss=blend


## Launch + robust save

In [15]:
import gc, time, torch, os, glob, shutil

trainer = AdvancedTrainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized_ds,
    data_collator    = data_collator,
    loss_mode        = LOSS_MODE,
    topk_min         = TOPK_MIN,
    blend_alpha      = BLEND_ALPHA,
    branch_logprob   = BRANCH_LOGPROB,
    warmup_mean_steps= WARMUP_MEAN_STEPS,
    stratified_order = stratified_order,
)

_ckpts = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = bool(_ckpts) and not SMOKE_TEST
print(f"{'Resuming' if resume else 'Fresh start'} in {args.output_dir}")

torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_err = None
try:
    trainer.train(resume_from_checkpoint=resume)
    print(f"Training done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e
    print(f"[TRAIN ERROR after {(time.time()-t0)/60:.1f} min] {type(e).__name__}: {e}")

print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
print(f"[loss-counters] nonfinite_logit_steps={trainer._c['nonfinite']} dead_steps={trainer._c['dead']}")
if trainer._c["dead"] > 0:
    print("[WARN] some steps fell back to zero loss -> lower TRAIN_MAX_LEN or batch size.")

def _save_adapter(dest):
    os.makedirs(dest, exist_ok=True)
    needed = ["adapter_config.json", "adapter_model.safetensors"]
    try:
        trainer.model.save_pretrained(dest); tokenizer.save_pretrained(dest)
    except Exception as e:
        print(f"[save] save_pretrained failed: {e}; trying trainer.save_model")
        try: trainer.save_model(dest); tokenizer.save_pretrained(dest)
        except Exception as e2: print(f"[save] save_model failed: {e2}")
    missing = [n for n in needed if not os.path.exists(os.path.join(dest, n))]
    if missing:
        cks = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
                     key=lambda p: int(p.rsplit("-", 1)[-1]))
        if cks:
            for fn in needed:
                sp = os.path.join(cks[-1], fn)
                if os.path.exists(sp): shutil.copy2(sp, os.path.join(dest, fn))
    have = {n: os.path.exists(os.path.join(dest, n)) for n in needed}
    print(f"[save] {dest} -> {have}")
    return all(have.values())

ok = _save_adapter(SFT_ADAPTER_DIR)
print("Adapter saved + verified." if ok else "[save] WARNING: files missing.")
if SMOKE_TEST:
    print("\n[SMOKE] green if loss-dbg showed real >0 loss + no dead_steps. Set SMOKE_TEST=0 and rerun.")
if train_err is not None:
    raise train_err

Fresh start in outputs/minmax_run
[loss-dbg step~0] mode=blend loss=6.2181 unmasked=6458 nll[mn/mean/mx]=-0.00/0.43/18.30 nonfinite=0
[loss-dbg step~0] mode=blend loss=6.3105 unmasked=6832 nll[mn/mean/mx]=-0.00/0.46/16.24 nonfinite=0
[loss-dbg step~0] mode=blend loss=8.1862 unmasked=4642 nll[mn/mean/mx]=-0.00/0.39/19.67 nonfinite=0


Step,Training Loss
10,49.635600
20,39.483900
30,28.659200
40,22.092200
50,16.655700
60,13.217300
70,11.108500
80,9.844200
90,8.357200
100,7.188700


Training done in 160.5 min
PEAK VRAM: 76.5 GB / 102 GB
[loss-counters] nonfinite_logit_steps=0 dead_steps=0
[save] outputs/minmax_logprob_adapter -> {'adapter_config.json': True, 'adapter_model.safetensors': True}
Adapter saved + verified.


## Greedy sanity check

In [16]:
import torch
model.eval()
try: model.config.use_cache = True
except Exception: pass
_probe = raw_ds[0]["user"]
_msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": _probe}]
try:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
except TypeError:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
_inp = tokenizer(_txt, return_tensors="pt").to(model.device)
with torch.no_grad():
    _o = model.generate(**_inp, max_new_tokens=512, do_sample=False, temperature=None, top_p=None)
_gen = tokenizer.decode(_o[0][_inp["input_ids"].shape[1]:], skip_special_tokens=True)
print(_gen[:1200])
print("\nHAS_BOXED:", "\\boxed{" in _gen)
try: model.config.use_cache = False
except Exception: pass
model.train()

[transformers_modules._1.modeling_nemotron_h|WARNING]NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


We need to deduce the transformation by matching the example outputs.

Output 0: 01001110
0 0
1 1
2 0
3 0
4 1
5 1
6 1
7 0

Output 1: 00010110
0 0
1 0
2 0
3 1
4 0
5 1
6 1
7 0

Output 2: 01000110
0 0
1 1
2 0
3 0
4 0
5 1
6 1
7 0

Output 3: 11000010
0 1
1 1
2 0
3 0
4 0
5 0
6 1
7 0

Output 4: 10001110
0 1
1 0
2 0
3 0
4 1
5 1
6 1
7 0

Output 5: 01011111
0 0
1 1
2 0
3 1
4 1
5 1
6 1
7 1

Output 6: 00011101
0 0
1 0
2 0
3 1
4 1
5 1
6 0
7 1

Output 7: 11110011
0 1
1 1
2 1
3 1
4 0
5 0
6 1
7 1

Output 8: 10011111
0 1
1 0
2 0
3 1
4 1
5 1
6 1
7 1

Output 9: 11011111
0 1
1 1
2 0
3 1
4 1
5 1
6 1
7 1

Output bit columns (with bitsum as hash)
0 0101001001 4
1 1001100110 5


HAS_BOXED: False


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): NemotronHForCausalLM(
      (backbone): NemotronHModel(
        (embeddings): Embedding(131072, 2688)
        (layers): ModuleList(
          (0): NemotronHBlock(
            (norm): NemotronHRMSNorm()
            (mixer): NemotronHMamba2Mixer(
              (act): SiLUActivation()
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144)
              (in_proj): Linear(in_features=2688, out_features=10304, bias=False)
              (norm): MambaRMSNormGated()
              (out_proj): Linear(in_features=4096, out_features=2688, bias=False)
            )
          )
          (1): NemotronHBlock(
            (norm): NemotronHRMSNorm()
            (mixer): NemotronHMOE(
              (experts): ModuleList(
                (0-127): 128 x NemotronHMLP(
                  (up_proj): Linear(in_features=2688, out_features=1856, bias=False)
                  (down_proj): Linear(in_features=

## Package submission.zip

In [17]:
import json, shutil, zipfile, os, glob

needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = SFT_ADAPTER_DIR
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT

missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    cks = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
                 key=lambda p: int(p.rsplit("-", 1)[-1]))
    if cks:
        os.makedirs(src_dir, exist_ok=True)
        for fn in needed:
            sp = os.path.join(cks[-1], fn)
            if os.path.exists(sp): shutil.copy2(sp, os.path.join(src_dir, fn))
missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    raise FileNotFoundError(f"Adapter files {missing} missing in {src_dir}.")

os.makedirs(SUBMISSION_DIR, exist_ok=True)
for fn in needed:
    shutil.copy2(os.path.join(src_dir, fn), os.path.join(SUBMISSION_DIR, fn))

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in needed:
        zf.write(os.path.join(SUBMISSION_DIR, fn), fn)
print(f"submission.zip -> {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

submission.zip -> /kaggle/working/submission.zip  (33.1 MB)
